# 개선된 LSTM 학습 (BiLSTM + Temporal Attention)

> **Runtime → GPU 설정 권장**: Runtime > Change runtime type > T4 GPU

| 개선 사항 | 설명 |
|-----------|------|
| BiLSTM | 양방향 LSTM으로 시퀀스 표현력 강화 |
| Temporal Attention | 중요한 타임스텝에 집중 |
| WeightedRandomSampler | 클래스 불균형 처리 (subsampling 없이 전체 데이터 사용) |
| BCEWithLogitsLoss + pos_weight | 안정적인 손실 함수 |
| AdamW + CosineAnnealingWarmRestarts | 안정적인 학습률 스케줄 |
| Val 기반 threshold 탐색 | 최적 분류 임계값 자동 탐색 |

In [1]:
# [STEP 1] Google Drive 마운트 & 경로 설정
import os, sys

# ★ 본인 Drive 경로에 맞게 수정하세요
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/뉴스크롤링"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR  = DRIVE_PROJECT_PATH
    DATA_DIR  = os.path.join(BASE_DIR, "")
    MODEL_DIR = os.path.join(BASE_DIR, "models")
    print(f"[Drive 마운트 완료] BASE_DIR={BASE_DIR}")
except Exception:
    BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    DATA_DIR  = os.path.join(BASE_DIR, "")
    MODEL_DIR = os.path.join(BASE_DIR, "models")
    print(f"[로컬 환경] BASE_DIR={BASE_DIR}")

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"DATA_DIR  : {DATA_DIR}")
print(f"MODEL_DIR : {MODEL_DIR}")
print(f"dataset.npz 존재: {os.path.exists(os.path.join(DATA_DIR, 'dataset.npz'))}")

Mounted at /content/drive
[Drive 마운트 완료] BASE_DIR=/content/drive/MyDrive/뉴스크롤링
DATA_DIR  : /content/drive/MyDrive/뉴스크롤링/
MODEL_DIR : /content/drive/MyDrive/뉴스크롤링/models
dataset.npz 존재: True


In [2]:
# [STEP 2] 라이브러리 임포트 & 디바이스 설정
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version : {torch.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU : {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Device : MPS (Apple Silicon)")
else:
    DEVICE = torch.device("cpu")
    print("Device : CPU")
print(f"DEVICE = {DEVICE}")

PyTorch version : 2.10.0+cu128
GPU : Tesla T4
DEVICE = cuda


In [3]:
# [STEP 3] 데이터 로드
path = os.path.join(DATA_DIR, "dataset.npz")
assert os.path.exists(path), f"dataset.npz 가 없습니다: {path}"

data    = np.load(path)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train 상승 비율 : {y_train.mean():.3f}")
print(f"y_test  상승 비율 : {y_test.mean():.3f}")

X_train : (1250163, 20, 18)
X_test  : (317147, 20, 18)
y_train 상승 비율 : 0.477
y_test  상승 비율 : 0.474


In [4]:
# [STEP 4] 하이퍼파라미터
HIDDEN_SIZE     = 256    # 128 → 256
NUM_HEADS       = 4      # Multi-head attention
NUM_LAYERS      = 2
DROPOUT         = 0.3
EPOCHS          = 40     # 30 → 40
BATCH_SIZE      = 2048
LR              = 3e-4
PATIENCE        = 10     # 7 → 10
N_ENSEMBLE      = 3      # 앙상블 모델 수
LABEL_SMOOTHING = 0.05   # 레이블 스무딩 (과신 방지)
ENSEMBLE_SEEDS  = [42, 123, 777]

print("하이퍼파라미터 설정 완료")

하이퍼파라미터 설정 완료


In [5]:
# [STEP 5] 모델: BiLSTM + Multi-Head Temporal Attention
class MultiHeadTemporalAttention(nn.Module):
    def __init__(self, hidden_size, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = (hidden_size * 2) // num_heads
        self.attn      = nn.Linear(hidden_size * 2, num_heads)
        self.out_proj  = nn.Linear(hidden_size * 2, hidden_size * 2)

    def forward(self, lstm_out):
        B, T, D = lstm_out.shape
        scores  = self.attn(lstm_out)                                     # (B, T, H)
        weights = F.softmax(scores, dim=1)                                # (B, T, H)
        x       = lstm_out.view(B, T, self.num_heads, self.head_dim)      # (B, T, H, d)
        context = (weights.unsqueeze(-1) * x).sum(dim=1).view(B, D)      # (B, D)
        return self.out_proj(context)


class ImprovedLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=2, dropout=0.3, num_heads=4):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.lstm = nn.LSTM(
            hidden_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.attention = MultiHeadTemporalAttention(hidden_size, num_heads)
        self.norm    = nn.LayerNorm(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2     = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        context = self.norm(context)
        context = self.dropout(context)
        out = F.gelu(self.fc1(context))
        out = self.dropout(out)
        return self.fc2(out).squeeze(-1)


input_size = X_train.shape[2]
_dummy   = ImprovedLSTMClassifier(input_size, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, NUM_HEADS)
n_params = sum(p.numel() for p in _dummy.parameters())
print(f"input_size  : {input_size}")
print(f"파라미터 수 : {n_params:,}")

input_size  : 18
파라미터 수 : 3,031,849


In [6]:
# [STEP 6] 학습 함수
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_best_threshold(probs, labels):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.30, 0.71, 0.02):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f"  최적 threshold={best_t:.2f}  val macro-F1={best_f1:.4f}")
    return best_t


def train_single(X_tr, y_tr, X_val, y_val, X_test, device, seed):
    set_seed(seed)

    # WeightedRandomSampler: 클래스 균형 처리
    # pos_weight는 사용하지 않음 — Sampler와 중복 보정 방지
    class_counts   = np.bincount(y_tr.astype(int))
    sample_weights = np.where(y_tr == 1, class_counts[0] / class_counts[1], 1.0)
    sampler = WeightedRandomSampler(
        weights=torch.FloatTensor(sample_weights),
        num_samples=len(sample_weights), replacement=True,
    )

    pin = (device.type == "cuda")
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
        batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=pin,
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
        batch_size=BATCH_SIZE * 2, shuffle=False,
    )

    model     = ImprovedLSTMClassifier(input_size, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, NUM_HEADS).to(device)
    criterion = nn.BCEWithLogitsLoss()   # pos_weight 제거 — Sampler가 이미 균형 담당

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            # Label smoothing: 0 → 0.05, 1 → 0.95 (과신 방지)
            yb_s = (yb * (1 - LABEL_SMOOTHING) + (1 - yb) * LABEL_SMOOTHING).to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb_s)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                yb_s = (yb * (1 - LABEL_SMOOTHING) + (1 - yb) * LABEL_SMOOTHING).to(device)
                val_loss += criterion(model(xb.to(device)), yb_s).item()
        val_loss /= len(val_loader)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  [seed={seed}] Epoch {epoch+1:3d}/{EPOCHS}  "
                  f"train={total_loss/len(train_loader):.4f}  "
                  f"val={val_loss:.4f}  lr={lr_now:.1e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  → Early stopping at epoch {epoch+1} (best val={best_val_loss:.4f})")
                break

    model.load_state_dict(best_state)
    model.eval()

    def get_probs(X):
        probs = []
        ld = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=BATCH_SIZE * 2)
        with torch.no_grad():
            for (xb,) in ld:
                probs.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
        return np.concatenate(probs)

    return get_probs(X_val), get_probs(X_test), best_state


print("train_single 함수 정의 완료")

train_single 함수 정의 완료


In [7]:
# [STEP 7] 앙상블 학습 실행 (seed 3개 모델 평균)
n_val        = max(1, int(len(X_train) * 0.15))
X_tr, X_val = X_train[:-n_val], X_train[-n_val:]
y_tr, y_val = y_train[:-n_val], y_train[-n_val:]

print(f"학습={len(X_tr):,}  검증={len(X_val):,}  테스트={len(X_test):,}")
print(f"Up 비율 — 학습: {y_tr.mean():.3f}  검증: {y_val.mean():.3f}")

val_probs_list  = []
test_probs_list = []
best_states     = []

for i, seed in enumerate(ENSEMBLE_SEEDS):
    print(f"\n{'='*45}")
    print(f"  모델 {i+1}/{N_ENSEMBLE}  (seed={seed})")
    print(f"{'='*45}")
    val_p, test_p, state = train_single(X_tr, y_tr, X_val, y_val, X_test, DEVICE, seed)
    val_probs_list.append(val_p)
    test_probs_list.append(test_p)
    best_states.append(state)

# 앙상블 평균 확률로 threshold 탐색 및 최종 예측
avg_val_probs  = np.mean(val_probs_list, axis=0)
avg_test_probs = np.mean(test_probs_list, axis=0)

print("\n[앙상블 threshold 탐색]")
threshold = find_best_threshold(avg_val_probs, y_val)
preds = (avg_test_probs > threshold).astype(int)

save_path = os.path.join(MODEL_DIR, "lstm_improved.pt")
torch.save(best_states[-1], save_path)
print(f"\n모델 저장: {save_path}")

학습=1,062,639  검증=187,524  테스트=317,147
Up 비율 — 학습: 0.478  검증: 0.473

  모델 1/3  (seed=42)
  [seed=42] Epoch   1/40  train=0.6941  val=0.6915  lr=2.9e-04
  [seed=42] Epoch   5/40  train=0.6918  val=0.6923  lr=1.5e-04
  [seed=42] Epoch  10/40  train=0.6908  val=0.6909  lr=3.0e-04
  [seed=42] Epoch  15/40  train=0.6903  val=0.6902  lr=2.6e-04
  [seed=42] Epoch  20/40  train=0.6886  val=0.6914  lr=1.5e-04
  → Early stopping at epoch 22 (best val=0.6900)

  모델 2/3  (seed=123)
  [seed=123] Epoch   1/40  train=0.6934  val=0.6935  lr=2.9e-04
  [seed=123] Epoch   5/40  train=0.6915  val=0.6915  lr=1.5e-04
  [seed=123] Epoch  10/40  train=0.6904  val=0.6913  lr=3.0e-04
  [seed=123] Epoch  15/40  train=0.6900  val=0.6905  lr=2.6e-04
  [seed=123] Epoch  20/40  train=0.6883  val=0.6906  lr=1.5e-04
  [seed=123] Epoch  25/40  train=0.6856  val=0.6930  lr=4.5e-05
  → Early stopping at epoch 27 (best val=0.6902)

  모델 3/3  (seed=777)
  [seed=777] Epoch   1/40  train=0.6937  val=0.6923  lr=2.9e-04
  [seed

In [8]:
# [STEP 8] 평가
acc = accuracy_score(y_test, preds)
f1  = f1_score(y_test, preds, average="weighted")

print("=" * 50)
print("  최종 결과")
print("=" * 50)
print(f"  Accuracy : {acc:.4f}  (기존 LSTM: 0.5082)")
print(f"  F1-score : {f1:.4f}  (기존 LSTM: 0.5055)")
print(f"  Up 예측 비율: {preds.mean():.3f}  (실제: {y_test.mean():.3f})")
print()
print(classification_report(y_test, preds, target_names=["Down", "Up"]))

  최종 결과
  Accuracy : 0.5109  (기존 LSTM: 0.5082)
  F1-score : 0.5002  (기존 LSTM: 0.5055)
  Up 예측 비율: 0.352  (실제: 0.474)

              precision    recall  f1-score   support

        Down       0.53      0.65      0.58    166939
          Up       0.48      0.36      0.41    150208

    accuracy                           0.51    317147
   macro avg       0.50      0.50      0.50    317147
weighted avg       0.50      0.51      0.50    317147



In [9]:
# [STEP 9] predictions.npz 저장
pred_path = os.path.join(DATA_DIR, "predictions.npz")

if os.path.exists(pred_path):
    existing         = dict(np.load(pred_path))
    existing["lstm"] = preds
    np.savez(pred_path, **existing)
    print(f"predictions.npz 업데이트: {pred_path}")
else:
    np.savez(pred_path, lstm=preds, y_test=y_test)
    print(f"predictions.npz 새로 저장: {pred_path}")

print(f"Down={( preds==0).sum():,}  Up={(preds==1).sum():,}")

predictions.npz 업데이트: /content/drive/MyDrive/뉴스크롤링/predictions.npz
Down=205,423  Up=111,724
